In [ ]:
!git clone https://github.com/TotallyNotMinh/Environmental-Sound-Recognition-Under-Unseen-Conditions.git
%cd /kaggle/working/Environmental-Sound-Recognition-Under-Unseen-Conditions
!git fetch origin
!git pull origin main

In [ ]:
!pip install -q h5py scipy opencv-python tqdm

In [ ]:
!torchrun --nproc_per_node=2 scripts/train_classifier.py \
        --data-path /kaggle/input/datasets/yousirui1/fsd50k/fsd50k \
        --batch-size 48 \
        --checkpoint-dir /kaggle/working/checkpoints/ \
        --encoder-lr 2e-4 \
        --head-lr 2e-4 

In [ ]:
# CMKD CNN teacher (EfficientNet-B0). --batch-size is the TOTAL batch (split across GPUs),
# 24 = CMKD's FSD50K CNN value. All other recipe values are paper defaults in the script.
!torchrun --nproc_per_node=2 scripts/train_cnn_classifier.py \
        --data-path /kaggle/input/datasets/yousirui1/fsd50k/fsd50k \
        --batch-size 24 \
        --checkpoint-dir /kaggle/working/checkpoints/fsd50k_cnn_b0/

In [ ]:
# CMKD CNN -> AST-Base knowledge distillation (run after the CNN teacher cell above).
# --batch-size is the TOTAL batch (split across GPUs), 12 = CMKD's FSD50K AST value.
# Smoke test first (no dataset/teacher needed):
#   !python scripts/train_kd.py --mock --num-epoch 2 --num-workers 0
!torchrun --nproc_per_node=2 scripts/train_kd.py \
        --data-path /kaggle/input/datasets/yousirui1/fsd50k/fsd50k \
        --teacher-checkpoint /kaggle/working/checkpoints/fsd50k_cnn_b0/best_cnn.pth \
        --batch-size 12 \
        --checkpoint-dir /kaggle/working/checkpoints/fsd50k_kd_ast_base/